# Lab 2: Dream locally, then dream again

**Cost: about $3 to $4 with the default Haiku 5.5 models (each cell prints an estimate first). Time: 35 minutes.**

You will run two dreams, the way a close team would after each close:

1. After the **September** close: dream over everything from the November 2025 to September 2026 closes (meetings, ERP exports,
   memos, sessions). This builds the playbook from nothing.
2. After the **October** close: dream over only October's sources, **into the same playbook**. Watch what changes.

What a dream does (`close_dreamer/dream.py`):

```
snapshot playbook ─► Phase 1: one analyzer per source file (verify, settle, enrich, organize), a close period at a
                   time: discussion and policy first, then the ERP evidence that confirms or corrects it
                 ─► Phase 2: one organizer (merge duplicates, fix links, complete _index.md)
                 ─► lint (shape and safety) + diff against the snapshot
```

The analyzer and organizer prompts are adapted from Anthropic's cookbook (`src/close_dreamer/prompts/`). Python plays
the cookbook's lead agent: it runs the phases itself.

**Setup:** copy `.env.example` to `.env` in the repo root and set `ANTHROPIC_API_KEY`. The `.env` file is gitignored.

In [ ]:
from pathlib import Path

from close_dreamer import config, memory
from close_dreamer import transcripts as tx
from close_dreamer.dream import dream, estimate

PLAYBOOK = config.OUTPUT / "lab-playbook"      # a separate folder so the lab doesn't touch the CLI's playbook
models = config.Models()
print(models)
all_t = tx.load(config.CORPUS)
import yaml

# The planted secret and injected instructions (data/ground_truth.yaml): lint proves they stayed out of memory.
truth = yaml.safe_load((config.DATA / "ground_truth.yaml").read_text(encoding="utf-8"))
probes = dict(extra_secrets=[truth["secret"]["text"]],
              extra_injections=[i["text"] for i in truth["injected_instructions"]])

## Dream 1: after the September close

In [ ]:
first = tx.select(all_t, through="2026-09")
print([t.name for t in first])
print(estimate(first, PLAYBOOK, models)[1])

In [ ]:
# Paid cell. Budget is a hard ceiling: the run stops if list-price spend passes it.
client = config.make_client()
# source_bytes is the record the playbook covers: the organizer aims for a playbook under about 30% of it.
r1 = dream(client, first, PLAYBOOK, config.SNAPSHOTS, models=models, budget_usd=3.00,
           label="lab-before-sep", source_bytes=sum(t.size for t in first), **probes)
print(r1.report())

### Look at what it built
Start where a future session starts: `_index.md`.

In [ ]:
print((PLAYBOOK / "_index.md").read_text(encoding="utf-8"))

In [ ]:
for p in sorted(PLAYBOOK.rglob("*.md")):
    print(f"{p.stat().st_size:>6,}  {p.relative_to(PLAYBOOK).as_posix()}")

In [ ]:
# Pick a page to read. Try the one about the intercompany cutoff or Allied Fastener.
import re
page = next((p for p in PLAYBOOK.rglob("*.md") if re.search(r"allied|uom", p.name, re.I)), None)
print(page.read_text(encoding="utf-8") if page else "No Allied page; grep for it instead.")

Check against Lab 1: is the Allied root cause recorded as **confirmed** (September) rather than a hunch (August)? Is
the Crestline reclass recorded as **posted on 9/3** (from the JE register) rather than parked? Is the accrual threshold
$25,000 with $10,000, $5,000 and $2,500 as history? Did the Houston count shortage and the Kansas City
TECO problem land on topic pages with their causes and fixes, rather than on a page per export?

## Dream 2: after the October close, into the same playbook

In [ ]:
second = tx.select(all_t, through="2026-10", after="2026-09")
print([t.name for t in second])
print(estimate(second, PLAYBOOK, models)[1])

In [ ]:
# The organizer is told which pages this dream touched, so it spends its turns there.
r2 = dream(client, second, PLAYBOOK, config.SNAPSHOTS, models=models, budget_usd=2.00,
           label="lab-before-oct", source_bytes=sum(t.size for t in first + second), **probes)
print(r2.report())

### What changed
This is the heart of the lab. The diff shows the playbook learning from one more close.

In [ ]:
print(memory.diff(r2.snapshot, PLAYBOOK)[:8000])

In [ ]:
# Superseded facts should be dated, not silently overwritten. Look for history lines and the changed rules.
for p in sorted(PLAYBOOK.rglob("*.md")):
    for line in p.read_text(encoding="utf-8").splitlines():
        if re.search(r"history|supersed|previously|was |BD2|ZCLS|9\.15|1300|1800004890|5000034412", line, re.I):
            print(f"{p.name:<40} {line.strip()[:120]}")

## Safety and shape

In [ ]:
print(memory.lint(PLAYBOOK, source_bytes=sum(t.size for t in first + second), **probes).summary())

## What you just saw

- **Correct:** the October dream replaced the IC cutoff, rate type, R-4410 standard and plants in scope, and kept the
  old values as dated history (check the diff; if it overwrote without history, that is a finding worth writing down).
- **Settle with evidence:** did the Allied page move from "fix pending" to "fixed 10/14, cleared 11/5" because of the
  change log and JE register? Did the Stuttgart fee get recorded as parked after cutoff, despite what Jonas said?
- **Build a representation:** small linked topic pages and an index instead of 129 files,
  well under a third of the record's size.
- **Fill gaps across sessions:** Allied's story and Monterrey's recurring IC posting error appear as single pages,
  though no single session saw the whole picture.
- **Safety:** lint shows whether the planted credential or injected rule leaked into memory.

**Try next:** rerun Dream 2 with `instructions="Keep a single 'current close rules' page at the top of the index"`
(the article's *focus on a goal* operation) and compare. Then run Lab 3 to measure whether any of this helps.